# Зачет по МФК «Математическая статистика и анализ данных»

Датасет содержит внутренние авиарейсы из аэропортов Нью-Йорка за 2013 год. Ниже я удаляю строки с пропущенными значениями и дальше работаю уже с очищенной таблицей.


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error, silhouette_score
from sklearn.cluster import KMeans


In [ ]:
df = pd.read_csv("flights_NY.csv")


In [ ]:
df.head(100)


In [ ]:
df = df.dropna()
df.shape


## 1. Вероятность задержки вылета

Выбираю авиакомпании, которые использовали больше 200 разных самолетов. Вероятность задержки оцениваю как долю рейсов, у которых `dep_delay > 0`.


In [ ]:
result = df.groupby('carrier').agg(
    n_planes=('tailnum', 'nunique'),
    flights=('flight', 'size'),
    delay_probability=('dep_delay', lambda x: (x > 0).mean())
)

result = result[result['n_planes'] > 200]
result = result.sort_values('n_planes', ascending=False)
result


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(result.index, result['n_planes'], alpha=0.7, label='planes')
plt.plot(result.index, result['delay_probability'], color='red', marker='o', label='delay probability')
plt.xlabel('carrier')
plt.ylabel('value')
plt.title('Carriers with more than 200 planes')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Наибольшая оценка вероятности задержки получилась у авиакомпании **WN**. При этом сортировка на графике идет по числу самолетов, как требовалось в задании.


## 2. Среднее время самолета в воздухе

Для каждого самолета считаю среднее значение `air_time`. После этого строю гистограмму и накладываю плотность нормального распределения с параметрами, оцененными по выборке.


In [ ]:
plane_air_time = df.groupby('tailnum')['air_time'].mean()

mu = plane_air_time.mean()
sigma = plane_air_time.std(ddof=1)
interval = stats.norm.interval(0.95, loc=mu, scale=sigma)

pd.DataFrame({
    'value': [len(plane_air_time), mu, sigma, interval[0], interval[1]]
}, index=['n_planes', 'mu', 'sigma', 'low_95', 'high_95'])


In [ ]:
x = np.linspace(plane_air_time.min(), plane_air_time.max(), 500)
y = stats.norm.pdf(x, mu, sigma)

plt.figure(figsize=(9, 5))
plt.hist(plane_air_time, bins=45, density=True, alpha=0.6, label='data')
plt.plot(x, y, color='red', linewidth=2, label='normal density')
plt.axvline(mu, color='black', linestyle='--', label='mean')
plt.axvspan(interval[0], interval[1], color='red', alpha=0.12, label='95% interval')
plt.xlabel('mean air time')
plt.ylabel('density')
plt.title('Mean air time by plane')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Оцененное среднее время в воздухе около **157.5 минут**, стандартное отклонение около **75.8 минут**. Если принять нормальную модель, то для 95% самолетов среднее время лежит примерно от **8.9** до **306.1 минут**.


## 3. Сравнение WN и UA

Сначала считаю скорость каждого рейса как `distance / air_time * 60`, затем для каждого самолета внутри авиакомпании нахожу среднюю скорость. Сравниваю распределения средних скоростей для WN и UA.


In [ ]:
df['speed'] = df['distance'] / df['air_time'] * 60

speed_by_plane = df.groupby(['carrier', 'tailnum']).agg(
    mean_speed=('speed', 'mean'),
    n_flights=('flight', 'size')
).reset_index()

wn = speed_by_plane[speed_by_plane['carrier'] == 'WN']['mean_speed']
ua = speed_by_plane[speed_by_plane['carrier'] == 'UA']['mean_speed']

wn_ci = stats.t.interval(0.95, len(wn) - 1, loc=wn.mean(), scale=stats.sem(wn))
ua_ci = stats.t.interval(0.95, len(ua) - 1, loc=ua.mean(), scale=stats.sem(ua))

ks_test = stats.ks_2samp(wn, ua)
t_test = stats.ttest_ind(wn, ua, equal_var=False)

summary = pd.DataFrame({
    'carrier': ['WN', 'UA'],
    'n': [len(wn), len(ua)],
    'mean': [wn.mean(), ua.mean()],
    'ci_low': [wn_ci[0], ua_ci[0]],
    'ci_high': [wn_ci[1], ua_ci[1]]
})

tests = pd.DataFrame({
    'test': ['Kolmogorov-Smirnov', 'Welch t-test'],
    'statistic': [ks_test.statistic, t_test.statistic],
    'p_value': [ks_test.pvalue, t_test.pvalue]
})

display(summary)
display(tests)


In [ ]:
x = np.linspace(min(wn.min(), ua.min()), max(wn.max(), ua.max()), 500)
wn_density = stats.gaussian_kde(wn)(x)
ua_density = stats.gaussian_kde(ua)(x)

plt.figure(figsize=(9, 5))
plt.plot(x, wn_density, color='red', label='WN')
plt.plot(x, ua_density, color='blue', label='UA')
plt.axvline(wn.mean(), color='red', linestyle='--')
plt.axvline(ua.mean(), color='blue', linestyle='--')
plt.axvspan(wn_ci[0], wn_ci[1], color='red', alpha=0.15)
plt.axvspan(ua_ci[0], ua_ci[1], color='blue', alpha=0.15)
plt.xlabel('mean speed')
plt.ylabel('density')
plt.title('Mean speed distributions for WN and UA')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Распределения отличаются значимо: p-value критерия Колмогорова-Смирнова очень мало. Средняя скорость у UA выше, а доверительные интервалы для средних WN и UA не пересекаются.


## 4. Регрессия средней дальности по средней скорости

Для каждого самолета считаю среднюю скорость и среднюю дальность рейса. Обычная линейная модель показывает общий рост, но зависимость нелинейная, поэтому использую полиномиальные признаки.


In [ ]:
plane_stats = df.groupby('tailnum').agg(
    mean_speed=('speed', 'mean'),
    mean_distance=('distance', 'mean'),
    n_flights=('flight', 'size')
).reset_index()

quality = []

for degree in [1, 2, 3]:
    model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())
    model.fit(plane_stats[['mean_speed']], plane_stats['mean_distance'])
    pred = model.predict(plane_stats[['mean_speed']])
    quality.append([degree, r2_score(plane_stats['mean_distance'], pred), mean_squared_error(plane_stats['mean_distance'], pred) ** 0.5])

quality = pd.DataFrame(quality, columns=['degree', 'R2', 'RMSE'])
quality


In [ ]:
model = make_pipeline(PolynomialFeatures(degree=3, include_bias=False), LinearRegression())
model.fit(plane_stats[['mean_speed']], plane_stats['mean_distance'])

x = np.linspace(plane_stats['mean_speed'].min(), plane_stats['mean_speed'].max(), 400)
y = model.predict(pd.DataFrame({'mean_speed': x}))

plt.figure(figsize=(9, 5))
plt.scatter(plane_stats['mean_speed'], plane_stats['mean_distance'], s=14, alpha=0.35, label='planes')
plt.plot(x, y, color='red', linewidth=2, label='polynomial regression')
plt.xlabel('mean speed')
plt.ylabel('mean distance')
plt.title('Mean distance vs mean speed')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Полином третьей степени дает лучший результат среди трех проверенных вариантов. По графику видно, что самолеты с большей средней скоростью обычно летают на более дальние расстояния.


## 5. Кластеризация самолетов

Кластеризую самолеты по двум признакам: средней скорости и числу выполненных рейсов. Перед KMeans признаки стандартизируются, потому что они измеряются в разных шкалах.


In [ ]:
cluster_data = plane_stats[['tailnum', 'mean_speed', 'n_flights']].copy()

scaler = StandardScaler()
X = scaler.fit_transform(cluster_data[['mean_speed', 'n_flights']])

scores = []
for k in range(2, 7):
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
    scores.append([k, silhouette_score(X, labels)])

scores = pd.DataFrame(scores, columns=['k', 'silhouette'])
scores


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
cluster_data['cluster'] = kmeans.fit_predict(X)

cluster_summary = cluster_data.groupby('cluster').agg(
    n=('tailnum', 'size'),
    mean_speed=('mean_speed', 'mean'),
    mean_flights=('n_flights', 'mean'),
    min_flights=('n_flights', 'min'),
    max_flights=('n_flights', 'max')
)

cluster_summary


In [ ]:
plt.figure(figsize=(9, 5))
for cluster in sorted(cluster_data['cluster'].unique()):
    part = cluster_data[cluster_data['cluster'] == cluster]
    plt.scatter(part['mean_speed'], part['n_flights'], s=16, alpha=0.6, label=f'cluster {cluster}')

plt.xlabel('mean speed')
plt.ylabel('number of flights')
plt.title('Plane clusters')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


По коэффициенту силуэта удобно взять 3 кластера. Один кластер содержит основную массу самолетов с умеренным числом рейсов, второй выделяет более медленные самолеты, третий — наиболее активно использовавшиеся борта с большим числом рейсов.


## Итог

В работе выполнены все пять заданий. Самая большая вероятность задержки среди крупных авиакомпаний получилась у WN. Средние скорости WN и UA различаются статистически значимо. Средняя дальность рейса связана со средней скоростью самолета, а кластеризация отделяет группы самолетов по скорости и интенсивности использования.
